In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn as sb
import os
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/BDKR81FL_converted (csv format).csv')

In [ ]:
print(df.columns.tolist())


['CASEID', 'BIDX', 'V000', 'V001', 'V002', 'V003', 'V004', 'V005', 'V006', 'V007', 'V008', 'V008A', 'V009', 'V010', 'V011', 'V012', 'V013', 'V014', 'V015', 'V016', 'V017', 'V018', 'V019', 'V019A', 'V020', 'V021', 'V022', 'V023', 'V024', 'V025', 'V026', 'V027', 'V028', 'V029', 'V030', 'V031', 'V032', 'V034', 'V040', 'V042', 'V044', 'V045A', 'V045B', 'V045C', 'V046', 'V047', 'V048', 'V101', 'V102', 'V103', 'V104', 'V105', 'V105A', 'V106', 'V107', 'V113', 'V115', 'V116', 'V119', 'V120', 'V121', 'V122', 'V123', 'V124', 'V125', 'V127', 'V128', 'V129', 'V130', 'V131', 'V133', 'V134', 'V135', 'V136', 'V137', 'V138', 'V139', 'V140', 'V141', 'V149', 'V150', 'V151', 'V152', 'V153', 'AWFACTT', 'AWFACTU', 'AWFACTR', 'AWFACTE', 'AWFACTW', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V166', 'V167', 'V168', 'V169A', 'V169B', 'V169C', 'V170', 'V171A', 'V171B', 'V172', 'V173', 'V174M', 'V174Y', 'V175', 'V176', 'V177', 'V190', 'V191', 'V190A', 'V191A', 'ML101', 'V201', 'V202', 'V203', 'V204'

In [ ]:
haz_candidates = [
    c for c in df.columns
    if c.lower().endswith("70") or "haz" in c.lower() or "hc70" in c.lower()
]

print("HAZ candidates:", haz_candidates)


HAZ candidates: ['V170', 'M70', 'HW70']


In [ ]:
age_candidates = [
    c for c in df.columns
    if c.lower() in ["b19", "age", "age_months", "child_age_months"]
       or "b19" in c.lower()
]

print("Age candidates:", age_candidates)


Age candidates: ['B19']


In [ ]:
import pandas as pd
import numpy as np

# df = pd.read_csv("BDHS_2022_KR.csv")   # or read_excel(...)

# -----------------------------
# 2A) RENAME VARIABLES (your list)
# -----------------------------

rename_candidates = {
    # Mandatory for stunting construction
    "B19":  "age_months",     # child age in months
    "HW70": "haz_raw",        # HAZ * 100

    # Wealth index / household wealth
    "V190": "wealth_index",

    # Mother's education
    "V106": "mother_edu",

    # Place of delivery / medical assistance at delivery
    "M15":  "place_delivery",         # place of delivery (common)
    "M3A":  "assist_doctor",           # delivery assistance: doctor (binary)
    "M3B":  "assist_nurse_midwife",    # nurse/midwife
    "M3C":  "assist_aux_midwife",      # auxiliary midwife
    "M3D":  "assist_trad_birth",       # traditional birth attendant
    "M3E":  "assist_other",            # other assistance

    # Toilet facility / improved sanitation
    "V116": "toilet_type",

    # Father's occupation
    "V705": "father_occ",     # partner's occupation (often V705)
    "V704": "father_worked",  # worked in last 12 months (partner)

    # Mother's BMI (maternal nutritional status)
    "V445": "mother_bmi_x10", # BMI * 10 (common DHS)
    "V437": "mother_height_mm",  # sometimes useful; optional

    # Birth order
    "BORD": "birth_order",

    # Television ownership
    "V121": "has_tv",

    # Mother's occupation
    "V714": "mother_occ",
    "V717": "mother_worked",  # worked in last 12 months

    # Birth interval (preceding birth interval)
    "B11":  "birth_interval_months",

    # Children ever born
    "V201": "children_ever_born",

    # Father's education
    "V701": "father_edu",

    # Child size at birth
    "M18":  "size_at_birth",

    # Desire for more children
    "V602": "desire_more_children",

    # Division (administrative division / region)
    "V024": "division",

    # Mother age
    "V012": "mother_age",

    # Reading newspaper
    "V157": "reads_newspaper",

    # Electricity access
    "V119": "has_electricity",

    # Weight (highly recommended)
    "V005": "sample_weight",

    # Optional IDs (useful but not required)
    "V001": "cluster_id",
    "V002": "hh_id",
    "BIDX": "child_line",
}

# Rename only columns that exist in your dataset
existing = set(df.columns)
rename_map = {k: v for k, v in rename_candidates.items() if k in existing}
missing = [k for k in rename_candidates.keys() if k not in existing]

df = df.rename(columns=rename_map)

print("Renamed (found in your file):")
for k, v in rename_map.items():
    print(f"  {k}  ->  {v}")

print("\nNot found in your file (we will fix names after you check columns):")
print(missing)


Renamed (found in your file):
  B19  ->  age_months
  HW70  ->  haz_raw
  V190  ->  wealth_index
  V106  ->  mother_edu
  M15  ->  place_delivery
  M3A  ->  assist_doctor
  M3B  ->  assist_nurse_midwife
  M3C  ->  assist_aux_midwife
  M3D  ->  assist_trad_birth
  M3E  ->  assist_other
  V116  ->  toilet_type
  V705  ->  father_occ
  V704  ->  father_worked
  V445  ->  mother_bmi_x10
  V437  ->  mother_height_mm
  BORD  ->  birth_order
  V121  ->  has_tv
  V714  ->  mother_occ
  V717  ->  mother_worked
  B11  ->  birth_interval_months
  V201  ->  children_ever_born
  V701  ->  father_edu
  M18  ->  size_at_birth
  V602  ->  desire_more_children
  V024  ->  division
  V012  ->  mother_age
  V157  ->  reads_newspaper
  V119  ->  has_electricity
  V005  ->  sample_weight
  V001  ->  cluster_id
  V002  ->  hh_id
  BIDX  ->  child_line

Not found in your file (we will fix names after you check columns):
[]


In [ ]:
# STEP 2B: Filter rows

# Convert HAZ to WHO units
df["haz"] = df["haz_raw"] / 100.0
df.loc[df["haz_raw"].isin([9996, 9997, 9998, 9999]), "haz"] = np.nan

# Keep children aged 0–59 months
df = df[df["age_months"].between(0, 59, inclusive="both")]

# Keep only valid HAZ
df = df[df["haz"].notna()].copy()

print("After Step 2B (filtered rows):", df.shape)


After Step 2B (filtered rows): (4105, 1239)


In [ ]:
df[["age_months", "haz"]].describe()


,age_months,haz
count,4105.000000,4105.000000
mean,29.050426,-1.159069
std,17.643332,1.268285
min,0.000000,-5.810000
25%,13.000000,-1.950000
50%,29.000000,-1.200000
75%,45.000000,-0.390000
max,59.000000,5.670000


In [ ]:
# STEP 3A: list of variables to keep (renamed names)

keep_cols = [
    # required for analysis
    "age_months", "haz", "haz_raw",

    # survey weight + optional IDs (keep them; do not model with IDs)
    "sample_weight", "wt", "cluster_id", "hh_id", "child_line",

    # YOUR predictors
    "wealth_index",
    "mother_edu",
    "place_delivery",
    "toilet_type",
    "father_occ",
    "mother_bmi_x10",
    "birth_order",
    "has_tv",
    "mother_occ",
    "birth_interval_months",
    "children_ever_born",
    "father_edu",
    "size_at_birth",
    "desire_more_children",
    "division",
    "mother_age",
    "reads_newspaper",
    "has_electricity",

    # delivery assistance (if you kept them in Step 2A)
    "assist_doctor",
    "assist_nurse_midwife",
    "assist_aux_midwife",
    "assist_trad_birth",
    "assist_other"
]


In [ ]:
# Keep only existing columns (prevents KeyError)
existing_keep_cols = [c for c in keep_cols if c in df.columns]

# Report missing (so you know what didn't exist)
missing_keep_cols = [c for c in keep_cols if c not in df.columns]
print("Missing columns (not in your df):", missing_keep_cols)

# Apply keep
df3 = df[existing_keep_cols].copy()

print("Before Step 3:", df.shape)
print("After Step 3:", df3.shape)
print("Columns kept:", df3.columns.tolist())


Missing columns (not in your df): ['wt']
Before Step 3: (4105, 1239)
After Step 3: (4105, 30)
Columns kept: ['age_months', 'haz', 'haz_raw', 'sample_weight', 'cluster_id', 'hh_id', 'child_line', 'wealth_index', 'mother_edu', 'place_delivery', 'toilet_type', 'father_occ', 'mother_bmi_x10', 'birth_order', 'has_tv', 'mother_occ', 'birth_interval_months', 'children_ever_born', 'father_edu', 'size_at_birth', 'desire_more_children', 'division', 'mother_age', 'reads_newspaper', 'has_electricity', 'assist_doctor', 'assist_nurse_midwife', 'assist_aux_midwife', 'assist_trad_birth', 'assist_other']


In [ ]:
# Optional: if you only need 'haz' (not haz_raw)
df3 = df3.drop(columns=["haz_raw"], errors="ignore")


In [ ]:
[c for c in df3.columns if "haz" in c.lower()]


['haz']

In [ ]:
"haz_raw" in df3.columns


False

In [ ]:
df3["haz"].describe()


,haz
count,4105.000000
mean,-1.159069
std,1.268285
min,-5.810000
25%,-1.950000
50%,-1.200000
75%,-0.390000
max,5.670000


In [ ]:
df3.to_csv("bdhs2022_step3_keptvars.csv", index=False)
df3.to_excel("bdhs2022_step3_keptvars.xlsx", index=False)
print("Saved Step 3 output.")


Saved Step 3 output.


In [ ]:
df3.shape


(4105, 29)

In [ ]:
len(df3.columns)


29

In [ ]:
for i, col in enumerate(df3.columns, start=1):
    print(i, col)


1 age_months
2 haz
3 sample_weight
4 cluster_id
5 hh_id
6 child_line
7 wealth_index
8 mother_edu
9 place_delivery
10 toilet_type
11 father_occ
12 mother_bmi_x10
13 birth_order
14 has_tv
15 mother_occ
16 birth_interval_months
17 children_ever_born
18 father_edu
19 size_at_birth
20 desire_more_children
21 division
22 mother_age
23 reads_newspaper
24 has_electricity
25 assist_doctor
26 assist_nurse_midwife
27 assist_aux_midwife
28 assist_trad_birth
29 assist_other


In [ ]:
haz_col = df3["haz"]
df3 = df3.drop(columns=["haz"])
df3["haz"] = haz_col


In [ ]:
df3.to_csv("bdhs2022_step3_keptvars.csv", index=False)
df3.to_excel("bdhs2022_step3_keptvars.xlsx", index=False)
print("Saved Step 3 output.")


Saved Step 3 output.


In [ ]:
for i, col in enumerate(df3.columns, start=1):
    print(i, col)

1 age_months
2 sample_weight
3 cluster_id
4 hh_id
5 child_line
6 wealth_index
7 mother_edu
8 place_delivery
9 toilet_type
10 father_occ
11 mother_bmi_x10
12 birth_order
13 has_tv
14 mother_occ
15 birth_interval_months
16 children_ever_born
17 father_edu
18 size_at_birth
19 desire_more_children
20 division
21 mother_age
22 reads_newspaper
23 has_electricity
24 assist_doctor
25 assist_nurse_midwife
26 assist_aux_midwife
27 assist_trad_birth
28 assist_other
29 haz


In [ ]:
df3.to_excel("bdhs2022_step3_final.xlsx", index=False)


In [ ]:
import numpy as np
import pandas as pd

df4 = df3.copy()
print(df4.shape)


(4105, 29)


In [ ]:
df4["stunted"] = (df4["haz"] < -2).astype(int)


In [ ]:
df4["stunted"].value_counts(dropna=False), df4["stunted"].mean()


(stunted
 0    3139
 1     966
 Name: count, dtype: int64,
 np.float64(0.23532277710109623))

In [ ]:
if "sample_weight" in df4.columns:
    df4["wt"] = df4["sample_weight"] / 1_000_000
else:
    print("Warning: sample_weight column not found. wt not created.")


In [ ]:
if "mother_bmi_x10" in df4.columns:
    df4["mother_bmi"] = df4["mother_bmi_x10"] / 10.0


In [ ]:
if "mother_bmi" in df4.columns:
    df4["mother_bmi"].describe()


In [ ]:
df4["mother_bmi"].describe()


,mother_bmi
count,4103.000000
mean,231.541506
std,42.006658
min,130.000000
25%,200.250000
50%,228.000000
75%,257.250000
max,437.100000


In [ ]:
# Check if both columns exist
print("mother_bmi_x10 in df4:", "mother_bmi_x10" in df4.columns)
print("mother_bmi in df4:", "mother_bmi" in df4.columns)

# Compare values (only where both are not missing)
check_df = df4[["mother_bmi_x10", "mother_bmi"]].dropna().copy()
check_df["recomputed_bmi"] = check_df["mother_bmi_x10"] / 10

# Check max difference
print("Max difference:", (check_df["mother_bmi"] - check_df["recomputed_bmi"]).abs().max())

mother_bmi_x10 in df4: True
mother_bmi in df4: True
Max difference: 0.0


# new recode codes.

In converting **df4** to **df_ml**, I restrict the sample to children who have valid delivery assistance information. This means I keep only observations where delivery-related variables are recorded and remove cases where that information is missing. I follow the approach used in the **BDHS 2022 report**, which analyzes delivery indicators only for births with available data. This keeps the dataset consistent with BDHS definitions and ensures that delivery variables in the modeling dataset are properly defined.






* I first examined the raw distribution of each variable in the master dataset (`df4`) to understand coding, frequencies, and missing values.
* I interpreted DHS value labels to ensure the recoding aligns with BDHS conceptual definitions.
* I designed and implemented the recode logic in `df4`, preserving valid missing values and appropriately grouping categories.
* I applied the same finalized recode logic to the modeling dataset (`df_ml`) to maintain consistency.
* I verified the distributions after recoding and removed the original raw variable from `df_ml`, keeping only the modeling-ready version.




In [ ]:
# Step 1: Restrict to children with valid delivery assistance info
df_ml = df4[df4["assist_doctor"].notna()].copy()

# Check new sample size
print("Original sample size:", df4.shape[0])
print("Restricted sample size:", df_ml.shape[0])

Original sample size: 4105
Restricted sample size: 2491


In [ ]:
# Create mother_bmi in df_ml directly
df_ml["mother_bmi"] = df_ml["mother_bmi_x10"] / 10.0

In [ ]:
df_ml = df_ml.drop(columns=["mother_bmi_x10"])

In [ ]:
print("mother_bmi in df_ml:", "mother_bmi" in df_ml.columns)
print("mother_bmi_x10 in df_ml:", "mother_bmi_x10" in df_ml.columns)
print(df_ml["mother_bmi"].describe())

mother_bmi in df_ml: True
mother_bmi_x10 in df_ml: False
count    2490.000000
mean      227.289518
std        41.033945
min       130.000000
25%       196.500000
50%       223.000000
75%       251.875000
max       422.700000
Name: mother_bmi, dtype: float64


In [ ]:
print("df_ml exists:", "df_ml" in globals())
print(df_ml.shape)

df_ml exists: True
(2491, 31)


In [ ]:
for var in [
    "mother_occ",
    "father_occ",
    "division",
    "mother_edu",
    "father_edu"
]:
    print("====", var, "====")
    print(df4[var].value_counts())
    print()

==== mother_occ ====
mother_occ
0    3110
1     995
Name: count, dtype: int64

==== father_occ ====
father_occ
8.0     1182
7.0      859
3.0      662
1.0      469
5.0      425
4.0      352
0.0       73
6.0       12
9.0       12
98.0       7
Name: count, dtype: int64

==== division ====
division
2    714
3    589
8    533
5    493
1    480
7    461
4    438
6    397
Name: count, dtype: int64

==== mother_edu ====
mother_edu
2    2139
1     938
3     775
0     253
Name: count, dtype: int64

==== father_edu ====
father_edu
2.0    1399
1.0    1226
3.0     798
0.0     623
8.0       7
Name: count, dtype: int64



In [ ]:
# Keep a copy if you want
df4["father_occ_clean"] = df4["father_occ"].copy()

# Merge the extremely rare category
df4.loc[df4["father_occ_clean"] == 6, "father_occ_clean"] = 7

# Sanity check
print(df4["father_occ"].value_counts())
print(df4["father_occ_clean"].value_counts())

father_occ
8.0     1182
7.0      859
3.0      662
1.0      469
5.0      425
4.0      352
0.0       73
6.0       12
9.0       12
98.0       7
Name: count, dtype: int64
father_occ_clean
8.0     1182
7.0      871
3.0      662
1.0      469
5.0      425
4.0      352
0.0       73
9.0       12
98.0       7
Name: count, dtype: int64


In [ ]:
print(df4.shape)
print("father_occ_clean in df4:", "father_occ_clean" in df4.columns)

(4105, 33)
father_occ_clean in df4: True


In [ ]:
print("df_ml exists:", "df_ml" in globals())
if "df_ml" in globals():
    print("df_ml shape:", df_ml.shape)
    print("father_occ in df_ml:", "father_occ" in df_ml.columns)
    print("Columns containing 'occ':", [c for c in df_ml.columns if "occ" in c])

df_ml exists: True
df_ml shape: (2491, 31)
father_occ in df_ml: True
Columns containing 'occ': ['father_occ', 'mother_occ']


In [ ]:
import numpy as np
import pandas as pd

# 1) Create father_occ_clean in df_ml (df_ml-only)
df_ml["father_occ_clean"] = pd.to_numeric(df_ml["father_occ"], errors="coerce")

# 2) Merge rare category 6 -> 7
df_ml.loc[df_ml["father_occ_clean"] == 6, "father_occ_clean"] = 7

# 3) Merge ultra-rare categories 9 and 98 -> 99 (other)
df_ml["father_occ_clean"] = df_ml["father_occ_clean"].replace({9: 99, 98: 99})

# 4) Sanity checks
print("father_occ_clean in df_ml:", "father_occ_clean" in df_ml.columns)
print(df_ml["father_occ_clean"].value_counts(dropna=False))
print("NaN in father_occ_clean:", df_ml["father_occ_clean"].isna().sum())

father_occ_clean in df_ml: True
father_occ_clean
8.0     746
7.0     552
3.0     386
1.0     279
5.0     249
4.0     203
0.0      48
NaN      19
99.0      9
Name: count, dtype: int64
NaN in father_occ_clean: 19


In [ ]:
# Merge ultra-rare categories into one group (99 = other)
df_ml["father_occ_clean"] = df_ml["father_occ_clean"].replace({
    9.0: 99,
    98.0: 99
})



In [ ]:
print(df_ml["father_occ_clean"].value_counts())

father_occ_clean
8.0     746
7.0     552
3.0     386
1.0     279
5.0     249
4.0     203
0.0      48
99.0      9
Name: count, dtype: int64


In [ ]:
sorted(df4["father_occ_clean"].dropna().unique())

[np.float64(0.0),
 np.float64(1.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(5.0),
 np.float64(7.0),
 np.float64(8.0),
 np.float64(9.0),
 np.float64(98.0)]

In [ ]:
print(df_ml["father_occ_clean"].value_counts(normalize=True))

father_occ_clean
8.0     0.301780
7.0     0.223301
3.0     0.156149
1.0     0.112864
5.0     0.100728
4.0     0.082120
0.0     0.019417
99.0    0.003641
Name: proportion, dtype: float64


In [ ]:
print("father_occ_clean NaN:", df_ml["father_occ_clean"].isna().sum())


father_occ_clean NaN: 19


In [ ]:
# Drop unnecessary occupation variables for ML
df_ml = df_ml.drop(columns=[
    "father_occ",
    "father_occ_clean_label",
    "father_occ_mca"
], errors="ignore")

# Sanity check
print("Remaining occupation-related columns:")
print([col for col in df_ml.columns if "father_occ" in col])

Remaining occupation-related columns:
['father_occ_clean']


In [ ]:
df_ml = df_ml.drop(columns=["father_occ_clean_label"], errors="ignore")

# Sanity check
print([col for col in df_ml.columns if "father_occ" in col])

['father_occ_clean']


father_occ_clean is for normal ml pipeline.

father_occ_mca is for doing mca later.

mother edu is fine, no need to recode. I just have to make sure to convert all of my categorical variable to strings or object. mother edu is not an exception. df4["mother_edu"] = df4["mother_edu"].astype(str)

In [ ]:
for var in [
    "assist_doctor",
    "assist_nurse_midwife",
    "assist_aux_midwife",
    "assist_trad_birth",
    "assist_other"
]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== assist_doctor ====
assist_doctor
NaN    1614
1.0    1399
0.0    1092
Name: count, dtype: int64

==== assist_nurse_midwife ====
assist_nurse_midwife
NaN    1614
1.0    1527
0.0     964
Name: count, dtype: int64

==== assist_aux_midwife ====
assist_aux_midwife
0.0    2464
NaN    1614
1.0      27
Name: count, dtype: int64

==== assist_trad_birth ====
assist_trad_birth
0.0    2489
NaN    1614
1.0       2
Name: count, dtype: int64

==== assist_other ====
assist_other
0.0    2489
NaN    1614
1.0       2
Name: count, dtype: int64



for medical, I'm changing everything and following bdhs 2022.

In [ ]:
for var in [
    "assist_doctor",
    "assist_nurse_midwife",
    "assist_aux_midwife",
    "assist_trad_birth",
    "assist_other"
]:
    print(var, "missing:", df_ml[var].isna().sum())

assist_doctor missing: 0
assist_nurse_midwife missing: 0
assist_aux_midwife missing: 0
assist_trad_birth missing: 0
assist_other missing: 0


In [ ]:
df_ml["skilled_birth"] = (
    (df_ml["assist_doctor"] == 1) |
    (df_ml["assist_nurse_midwife"] == 1) |
    (df_ml["assist_aux_midwife"] == 1)
).astype(int)

print(df_ml["skilled_birth"].value_counts())

skilled_birth
1    1707
0     784
Name: count, dtype: int64


skilled_birth = 1 → Skilled assistance  

skilled_birth = 0 → Non-skilled or no skilled assistance

skilled birth, it is oaky with my mca variable  too.

In [ ]:
df_ml = df_ml.drop(columns=[
    "assist_doctor",
    "assist_nurse_midwife",
    "assist_aux_midwife",
    "assist_trad_birth",
    "assist_other"
], errors='ignore')

dropped all unncessesary assists.

In [ ]:
print("df4 shape:", df4.shape)
print("df_ml shape:", df_ml.shape)

print("Missing in df4 place_delivery:", df4["place_delivery"].isna().sum())
print("Missing in df_ml place_delivery:", df_ml["place_delivery"].isna().sum())

df4 shape: (4105, 33)
df_ml shape: (2491, 27)
Missing in df4 place_delivery: 1614
Missing in df_ml place_delivery: 0


In [ ]:
for var in [
    "toilet_type",
    "place_delivery"
]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== toilet_type ====
toilet_type
22    964
12    832
21    807
23    579
97    406
13    202
11    156
14     59
43     55
15     22
31     17
42      4
96      2
Name: count, dtype: int64

==== place_delivery ====
place_delivery
NaN     1614
11.0     889
33.0     792
32.0     326
25.0     126
23.0     115
21.0      78
24.0      49
41.0      48
27.0      20
28.0      15
31.0      11
22.0      11
26.0       8
96.0       3
Name: count, dtype: int64



In [ ]:
df_ml["toilet_group"] = df_ml["toilet_type"].map({

    # Improved
    11: "improved",
    12: "improved",
    13: "improved",
    21: "improved",
    22: "improved",
    41: "improved",

    # Unimproved
    14: "unimproved",
    15: "unimproved",
    23: "unimproved",
    31: "unimproved",
    42: "unimproved",
    43: "unimproved",

    # Other / special
    96: "other",
    97: "other"
})

print(df_ml["toilet_group"].value_counts(dropna=False))

toilet_group
improved      1775
unimproved     428
other          288
Name: count, dtype: int64


In [ ]:
df_ml["toilet_improved"] = (df_ml["toilet_group"] == "improved").astype(int)

In [ ]:
df_ml = df_ml.drop(columns=["toilet_type"])

In [ ]:
df_ml = df_ml.drop(columns=["toilet_group"])

In [ ]:
print(df_ml["toilet_improved"].value_counts())

toilet_improved
1    1775
0     716
Name: count, dtype: int64


this binary toilet improved variable is for normal ml pipeline.

now recoding : place delivery variable.

In [ ]:
# Step 1: Define mapping (once)
delivery_map_3cat = {

    # Home
    11: 0,

    # Public sector
    21: 1, 22: 1, 23: 1, 24: 1,
    25: 1, 26: 1, 27: 1, 28: 1,

    # Private sector (NGO merged)
    31: 2, 32: 2, 33: 2, 36: 2,
    41: 2, 42: 2,

    # Other → merge into private
    96: 2
}

In [ ]:
# Step 2: Apply in df4 (source of truth)
df4["delivery_place_3cat"] = df4["place_delivery"].map(delivery_map_3cat)

# Verify in df4
df4["delivery_place_3cat"].value_counts(dropna=False)

,count
delivery_place_3cat,
NaN,1614
2.0,1180
0.0,889
1.0,422


In [ ]:
# Step 3: Apply same logic in df_ml
df_ml["delivery_place_3cat"] = df_ml["place_delivery"].map(delivery_map_3cat)

# Verify in df_ml
df_ml["delivery_place_3cat"].value_counts(dropna=False)

,count
delivery_place_3cat,
2,1180
0,889
1,422


In [ ]:
# Step 4: Drop raw variable from df_ml only
df_ml = df_ml.drop(columns=["place_delivery"])

In [ ]:
print("NaN in original (df4):", df4["place_delivery"].isna().sum())

NaN in original (df4): 1614


In [ ]:
print("NaN in original (df4):", df4["place_delivery"].isna().sum())
print("NaN after mapping (df_ml):", df_ml["delivery_place_3cat"].isna().sum())

NaN in original (df4): 1614
NaN after mapping (df_ml): 0


In [ ]:
print("df_ml shape:", df_ml.shape)

df_ml shape: (2491, 27)


In [ ]:
print("df_ml exists:", "df_ml" in globals())
print("df_ml shape:", df_ml.shape if "df_ml" in globals() else None)

# Ensure these ML features exist
for col in ["skilled_birth", "toilet_improved"]:
    print(col, "in df_ml:", col in df_ml.columns)

df_ml exists: True
df_ml shape: (2491, 27)
skilled_birth in df_ml: True
toilet_improved in df_ml: True


In [ ]:
print("df_mca exists:", "df_mca" in globals())

df_mca exists: False


In [ ]:
if "df_mca" in globals():
    del df_mca
    print("df_mca deleted.")
else:
    print("df_mca already not present.")

df_mca already not present.


In [ ]:
print("df_mca exists:", "df_mca" in globals())

df_mca exists: False


new variable recode here. has tv, has electricity, read, and rest

In [ ]:
for var in ["has_tv"]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== has_tv ====
has_tv
0    1875
1    1824
7     406
Name: count, dtype: int64



In [ ]:
df4.loc[:, "has_tv_clean"] = df4["has_tv"].replace({7: np.nan})

In [ ]:
df_ml["has_tv_clean"] = df_ml["has_tv"].replace({7: np.nan})

In [ ]:
df_ml["has_tv_clean"].value_counts(dropna=False)

,count
has_tv_clean,
0.0,1141
1.0,1063
NaN,287


In [ ]:
df_ml = df_ml.drop(columns=["has_tv"])

has electricity recode

In [ ]:
for var in ["has_electricity"]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== has_electricity ====
has_electricity
1    3635
7     406
0      64
Name: count, dtype: int64



In [ ]:
df4.loc[:, "has_electricity_clean"] = df4["has_electricity"].replace({7: np.nan})

In [ ]:
print([col for col in df_ml.columns if "119" in col or "elec" in col.lower()])

['has_electricity']


In [ ]:
df_ml["has_electricity_clean"] = df_ml["has_electricity"].replace({7: np.nan})
df_ml = df_ml.drop(columns=["has_electricity"])
df_ml["has_electricity_clean"].value_counts(dropna=False)

,count
has_electricity_clean,
1.0,2172
NaN,287
0.0,32


reads newspaper variable recode.

In [ ]:
for var in ["reads_newspaper"]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== reads_newspaper ====
reads_newspaper
0    3786
1     215
2     104
Name: count, dtype: int64



In [ ]:
df4.loc[:, "reads_newspaper_clean"] = df4["reads_newspaper"].replace({
    0: 0,
    1: 1,
    2: 1,
    3: 1
})

In [ ]:
df4["reads_newspaper_clean"].value_counts(dropna=False)

,count
reads_newspaper_clean,
0,3786
1,319


In [ ]:
df_ml["reads_newspaper_clean"] = df_ml["reads_newspaper"].replace({
    0: 0,
    1: 1,
    2: 1,
    3: 1
})

In [ ]:
df_ml["reads_newspaper_clean"].value_counts(dropna=False)

,count
reads_newspaper_clean,
0,2297
1,194


In [ ]:
df_ml = df_ml.drop(columns=["reads_newspaper"])

next is size at birth.

In [ ]:
for var in ["size_at_birth"]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== size_at_birth ====
size_at_birth
3.0    1908
NaN    1614
4.0     336
2.0     203
5.0      19
1.0      19
8.0       6
Name: count, dtype: int64



In [ ]:
df4.loc[:, "small_at_birth"] = df4["size_at_birth"].map({
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1
})

In [ ]:
df4["small_at_birth"].value_counts(dropna=False)

,count
small_at_birth,
0.0,2130
NaN,1620
1.0,355


In [ ]:
df_ml["small_at_birth"] = df_ml["size_at_birth"].map({
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1
})

In [ ]:
df_ml["small_at_birth"].value_counts(dropna=False)

,count
small_at_birth,
0.0,2130
1.0,355
NaN,6


In [ ]:
df_ml["small_at_birth"].value_counts(dropna=False)

,count
small_at_birth,
0.0,2130
1.0,355
NaN,6


In [ ]:
df_ml = df_ml.drop(columns=["size_at_birth"])

next is desire more children.

In [ ]:
for var in ["desire_more_children"]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== desire_more_children ====
desire_more_children
3    2038
1    1559
2     368
4     114
5      26
Name: count, dtype: int64



In [ ]:
df4.loc[:, "wants_more_children"] = df4["desire_more_children"].map({
    1: 1,
    3: 0,
    4: 0,
    5: 0
})

In [ ]:
df4["wants_more_children"].value_counts(dropna=False)

,count
wants_more_children,
0.0,2178
1.0,1559
NaN,368


In [ ]:
df_ml["wants_more_children"] = df_ml["desire_more_children"].map({
    1: 1,
    3: 0,
    4: 0,
    5: 0
})

In [ ]:
df_ml["wants_more_children"].value_counts(dropna=False)

,count
wants_more_children,
0.0,1229
1.0,1032
NaN,230


In [ ]:
df_ml = df_ml.drop(columns=["desire_more_children"])

In [ ]:
df_ml["delivery_place_3cat"].isna().sum()

np.int64(0)

now wealth index recode.

In [ ]:
for var in ["wealth_index"]:
    print("====", var, "====")
    print(df4[var].value_counts(dropna=False))
    print()

==== wealth_index ====
wealth_index
1    882
3    826
2    807
5    799
4    791
Name: count, dtype: int64



In [ ]:
df4.loc[:, "wealth_index_clean"] = df4["wealth_index"].astype("Int64")

In [ ]:
df4["wealth_index_clean"].value_counts(dropna=False).sort_index()

,count
wealth_index_clean,
1,882
2,807
3,826
4,791
5,799


In [ ]:
df_ml["wealth_index_clean"] = df_ml["wealth_index"].astype("Int64")

In [ ]:
df_ml = df_ml.drop(columns=["wealth_index"])

In [ ]:
df_ml.isna().sum().sort_values(ascending=False)

,0
birth_interval_months,964
has_electricity_clean,287
has_tv_clean,287
wants_more_children,230
father_occ_clean,19
father_edu,19
small_at_birth,6
mother_bmi,1
sample_weight,0
cluster_id,0


In [ ]:
recoded_vars = [
    "has_tv_clean",
    "has_electricity_clean",
    "small_at_birth",
    "wants_more_children",
    "father_edu",
    "father_occ_clean"

]

df_ml[recoded_vars].isna().sum()

,0
has_tv_clean,287
has_electricity_clean,287
small_at_birth,6
wants_more_children,230
father_edu,19
father_occ_clean,19


In [ ]:
print("df4 missing:", df4["small_at_birth"].isna().sum())
print("df_ml missing:", df_ml["small_at_birth"].isna().sum())

df4 missing: 1620
df_ml missing: 6


In [ ]:
import numpy as np
import pandas as pd

# 1) Replace DHS-style missing codes with NaN (numeric columns only)
missing_codes = {8, 9, 98, 99, 998, 999}

for col in df_ml.columns:
    if pd.api.types.is_numeric_dtype(df_ml[col]):
        df_ml.loc[df_ml[col].isin(missing_codes), col] = np.nan

# 2) Move outcome variables to the end (haz last)
end_cols = [c for c in ["stunted", "haz"] if c in df_ml.columns]
cols = [c for c in df_ml.columns if c not in end_cols] + end_cols
df_ml = df_ml[cols]

print("df_ml updated. Last columns:", df_ml.columns.tolist()[-len(end_cols):])
print("Shape:", df_ml.shape)

df_ml updated. Last columns: ['stunted', 'haz']
Shape: (2491, 27)


In [ ]:
print("Shape:", df_ml.shape)
print("Duplicate rows:", df_ml.duplicated().sum())
print("Index is unique:", df_ml.index.is_unique)

Shape: (2491, 27)
Duplicate rows: 0
Index is unique: True


In [ ]:
# Total number of columns
print("Number of columns:", len(df_ml.columns))

# Full column list
print("\nColumn names:")
for col in df_ml.columns:
    print(col)



Number of columns: 27

Column names:
age_months
sample_weight
cluster_id
hh_id
child_line
mother_edu
birth_order
mother_occ
birth_interval_months
children_ever_born
father_edu
division
mother_age
wt
mother_bmi
father_occ_clean
skilled_birth
toilet_improved
delivery_place_3cat
has_tv_clean
has_electricity_clean
reads_newspaper_clean
small_at_birth
wants_more_children
wealth_index_clean
stunted
haz


In [ ]:
# Check existence
print("haz in df_ml:", "haz" in df_ml.columns)
print("stunted in df_ml:", "stunted" in df_ml.columns)

# Check stunted distribution
print("\nStunted value counts:")
print(df_ml["stunted"].value_counts(dropna=False))


haz in df_ml: True
stunted in df_ml: True

Stunted value counts:
stunted
0.0    1908
1.0     583
Name: count, dtype: int64


In [ ]:
# Total missing values in entire dataframe
total_missing = df_ml.isna().sum().sum()
print("Total missing values in df_ml:", total_missing)

Total missing values in df_ml: 3161


In [ ]:
missing_count = df_ml.isna().sum()
missing_percent = (df_ml.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent
})

# Show only columns with missing values
missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

,Missing Count,Missing %
birth_interval_months,983,39.46
father_occ_clean,774,31.07
division,302,12.12
has_electricity_clean,287,11.52
has_tv_clean,287,11.52
wants_more_children,230,9.23
age_months,168,6.74
hh_id,82,3.29
father_edu,22,0.88
cluster_id,15,0.60


In [ ]:
# Categorical variables to fill with mode
cat_vars_to_fill = [
    "has_tv_clean",
    "has_electricity_clean",
    "wants_more_children",
    "father_edu",
    "father_occ_clean",
    "small_at_birth",
    "division"
]

# Fill missing values with mode
for col in cat_vars_to_fill:
    mode_value = df_ml[col].mode(dropna=True)[0]
    df_ml[col].fillna(mode_value, inplace=True)

print("Categorical missing values filled with mode.")

Categorical missing values filled with mode.


/tmp/ipykernel_591/4276657425.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_ml[col].fillna(mode_value, inplace=True)


In [ ]:
missing_count = df_ml.isna().sum()
missing_percent = (df_ml.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent
})

# Show only columns with missing values
missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

,Missing Count,Missing %
birth_interval_months,983,39.46
age_months,168,6.74
hh_id,82,3.29
cluster_id,15,0.60
birth_order,2,0.08
children_ever_born,2,0.08
mother_bmi,1,0.04


In [ ]:
# Save final analysis-ready dataset (df_ml)

df_ml.to_csv("bdhs2022_final_analysis_dataset_new.csv", index=False)
df_ml.to_excel("bdhs2022_final_analysis_dataset_new.xlsx", index=False)

print("Final dataset saved as:")
print("- bdhs2022_final_analysis_dataset_new.csv")
print("- bdhs2022_final_analysis_dataset_new.xlsx")
print("Final shape:", df_ml.shape)

Final dataset saved as:
- bdhs2022_final_analysis_dataset_new.csv
- bdhs2022_final_analysis_dataset_new.xlsx
Final shape: (2491, 27)


for paper, I will focus more on this matter.

In [ ]:
df_ml[df_ml['birth_order'] == 1]['birth_interval_months'].describe()

,birth_interval_months
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [ ]:
print("Min:", df_ml['mother_bmi'].min())
print("Max:", df_ml['mother_bmi'].max())

Min: 130.0
Max: 422.7
